### Problem Set "Portfolio Sorts" for BSc students

## General Notes on this PS
- The student is asked to go through the code thoroughly and take note of the setup and structure of the code
- After having done this, the student should have an understanding of these following points:
1. firstly all relevant libraries are imported
2. After that, parameters that determine the behavior of the code and the sorting algorithm are set, all in one central place
3. Furthermore the code employs parallelization of the computational tasks, to speed up the runtime of the code (see "multiprocessing as mp")
4. The whole logic of the code is setup in an object oriented way, i.e. through a class called "PortfolioSortsPipeline"
- In the last line of the code, an object of this class is created and the method (class function) "runPipeline" is called, to perform all steps, one after the other
- The object oriented structure of the code enables the user to modularize/split up the code in logical units, more easily debug singular steps and most importantly reuse the code in other projects
- Each step of the procedure is associated with one method of the class; the methods are called one after the other in the "runPipeline" function and are currently all commented out, so nothing will happen

## Tasks for students
- make sure you've installed all libraries needed
- make sure you've set a path for the "baseFolder" variable (code line 14)
- make sure the subfolders within your baseFolder exist
- make sure you've placed all input data that comes with this problem set in the baseFolder path you've just specified
- Throughout the code important steps have been removed and replaced with a block like this:
- ########## STUDENT TASK START ##########
- write code that does x...
- ########## STUDENT TASK END ##########
- complete the code inbetween such lines
- If the student feels confident about his solution, he can comment-in the corresponding function in the "runPipeline" method and run the pipeline
- After having completed all tasks in the code, the code should produce two files in the "resultsPath" folder (code line 16):
- PortfoliosOutcomePlusDiff.parquet which contains the time series of the sorted portfolios
- PortfoliosOutcomePlusDiffTest.parquet which contains the statistical tests of the time series of the sorted portfolios
- and should describe, analyze and intepret the results in written form/presentation
- **If you're having trouble, add logging statements and execute the code step by step, to see what's going on**

### Documentation

## Import statments
Some common libraries are used throghout this script

## Parameters
Some parameters are set, notably: <br>
Folders that point to where the input data, the logfile and the results lie <br>
Parameters directly relevant to the portfolio sorts algorithm, e.g. start and end date, sorting type, sortvariable names, weighing type etc. <br>

## Description of data
This problem set comes with the following input data: <br>
SPXConstituentsPrices.parquet --> Prices of the SPX constituents <br>
RiskFreeRatesFiltered.parquet --> Risk free rates <br>
CRAMMeasures.parquet --> Risk measures (sortvariables) <br>

## Code
# calcMonthlyExcessReturn()
This function loads prices and risk free rates from the baseFolder, where the data has to be put <br>
It further calculates monthly excess returns for each securityid (identifyer of stock) and offsets these returns by one month, to get the one month ahead excess return, which will later be the outcome variable for the portfolio sorts <br>
Input: SPXConstituentsPrices.parquet, RiskFreeRatesFiltered.parquet <br>
Output: RiskFreeRatesFilteredMonthly.parquet, SPXConstituentsPricesPrepared.parquet <br>

# mergeData()
This function merges the CRAM data (measures/sortvariables) and the recently calculated one month ahead excess returns together, based on the identifiyer SecurityID <br>
Input: SPXConstituentsPricesPrepared.parquet, CRAMMeasures.parquet <br>
Output: MergedInputData.parquet <br>

# getSortingDates()
This function determines the first available dates each month, which will later be used for sorting and filters the input data for relevant dates and columns <br>
Input: MergedInputData.parquet <br>
Output: MergedInputDataFiltered.parquet, SortingDates.parquet <br>

# buildSortedPortfolios()
This function performs the sorting algorithm. It takes all relevant parameters (variables) defined above, the input data and sorting dates. <br>
It iterates over each date in the date chunk <br>
For each date it collects information on missing data <br>
Breakpoints based on the sortvariable(s) are calculated. <br>
Instruments are sorted into portfolios based on these breakpoints <br>
Optionally, portfolios are filtered (excluded) <br>
Weights of each stock within each portfolio (marketcap or equally weighted) and the average outcome variable per portfolio are calculated <br>
the results are saved to pkl files <br>
Input: MergedInputDataFiltered.parquet, SortingDates.parquet <br>
Output: sortedPortfolioResults.pkl, sortedOutcomeResults.pkl, numberOfStocks.pkl, breakpointsDict.pkl, missingDataDict.pkl, missingRebalancingDatesList.pkl <br>

# convertOutcome()
This function converts the sortedOutcomeResults.pkl into a table and saves it into a parquet file. <br>
Input: sortedOutcomeResults.pkl <br>
Output: PortfoliosOutcome.parquet <br>

# calcDiffPortfolios()
This function calculates the diff portfolios based on the sorted portfolios depending on the type of sorting (univariate, bivariate) <br>
Input: PortfoliosOutcome.parquet <br>
Output: PortfoliosOutcomePlusDiff.parquet <br>

# performStatTest()
This function performs calculates the time series average and performes statistical tests on the sorted portfolio time series average outcome values. <br>
Input: PortfoliosOutcomePlusDiff.parquet <br>
Output: PortfoliosOutcomePlusDiffTest.parquet <br>

In [1]:
# Import libraries
import os
import time
import pickle
import pandas as pd
import numpy as np
from loguru import logger
from statsmodels.regression.linear_model import OLS
from statsmodels.tools.tools import add_constant

# Directories directly relevant to this script
baseFolder = os.path.join(os.getcwd(), 'Data') # Input data folder (relative to notebook directory)
#projectFolder = os.path.dirname(baseFolder)
logFilePath = os.path.join(baseFolder, 'logfilePortfolioSortsPipeline.log') # The log file goes here
resultsPath = os.path.join(baseFolder, 'solution') # Solutions go here

# Parameters
sortingType = "Univariate" # "Univariate", "BivariateDependent" or "BivariateIndependent"
sortVariable1 = "op_at" # name of the first sorting variable
sortVariable2 = "market_equity" # name of the second sorting variable
sortPercentilesVar1 = [30, 70] # percentiles for the split of sort variable 1
sortPercentilesVar2 = [30, 70] # percentiles for the split of sort variable 2
weightType = "MarketCapWeighted" # weighting scheme within the portfolios, "MarketCapWeighted" or "EquallyWeighted"

testmode = False # if test mode is set to true, the sorting of portfolios will only be applied to a small number of dates
log_results = False # Variable to control whether the results of the portfolio sorting are printed in the log

# FIXED PARAMETERS - DO NOT CHANGE
startDate = pd.to_datetime('01.01.2012', format='%d.%m.%Y') # start date for the procedure
endDate = pd.to_datetime('31.12.2021', format='%d.%m.%Y') # end date for the procedure
marketCapColumn = "MCap" # name of the column that contains the market capitalization
outcomeVariable = "return_month_ahead_excess" # name of the outcome variable
filterPortfolios = None # Optional parameter for filtering out sorted portfolios by index
backtrackingDays = 0 # Variable to handle the number of days of backtracking in case of no measure data for portfolio sorts, values other than 0 are not supported in this code version
lags = 6 # the number of lags for the Newey-West adjustment for the statistical tests

# class that handles all logic
class PortfolioSortsPipeline():

    def __init__(self):
        self.start_time = None
        self.end_time = None

    def start(self):
        # Set up logging
        self.start_time = time.time()
        if os.path.exists(logFilePath):
            os.remove(logFilePath)
        logger.add(logFilePath, level='INFO')
        logger.info(f"Start Time: {time.strftime('%Y-%m-%d %H:%M:%S', time.localtime(self.start_time))}")
        logger.info('Start portfolio sorts pipeline...')

    def clearSolutionsFolder(self):
        logger.info(f'Clearing solution folder at {resultsPath}...')
        if os.path.exists(resultsPath):
            for filename in os.listdir(resultsPath):
                file_path = os.path.join(resultsPath, filename)
                try:
                    if os.path.isfile(file_path) or os.path.islink(file_path):
                        os.unlink(file_path)
                    elif os.path.isdir(file_path):
                        os.rmdir(file_path)
                except Exception as e:
                    logger.error(f'Failed to delete {file_path}. Reason: {e}')
        else:
            os.makedirs(resultsPath)

    def end(self):
        self.end_time = time.time()
        duration_seconds = self.end_time - self.start_time
        duration_formatted = time.strftime('%H:%M:%S', time.gmtime(duration_seconds))
        logger.info('Portfolio sorts pipeline done!')
        logger.info(f"End Time: {time.strftime('%Y-%m-%d %H:%M:%S', time.localtime(self.end_time))}")
        logger.info(f'Duration: {duration_formatted}')

    def calcMonthlyExcessReturns(self):
        # File Paths
        prices_filepath = baseFolder + '/SPXConstituentsPrices.parquet'
        riskfree_filepath = baseFolder + '/RiskFreeRatesFiltered.parquet'
        riskfree_output = resultsPath + "/RiskFreeRatesFilteredMonthly.parquet"
        prices_output = resultsPath + "/SPXConstituentsPricesPrepared.parquet"

        # Load Prices and Risk-Free Rates
        pricesData = pd.read_parquet(prices_filepath)
        riskFreeRatesData = pd.read_parquet(riskfree_filepath)

        pricesData['date'] = pd.to_datetime(pricesData['date'])
        riskFreeRatesData['date'] = pd.to_datetime(riskFreeRatesData['date'])

        ########## STUDENT TASK START ##########
        # Produce two output files:
        #   (1) riskFreeRatesData -> riskfree_output (parquet, index=False)
        #   (2) pricesData        -> prices_output    (parquet, index=False)
        # pricesData must carry one column called 'return_month_ahead_excess'
        # holding each row's next-month excess return for the SPX constituent.

        # Filter risk-free rates to 30-day maturity and align them to calendar month-end
        rf_monthly = riskFreeRatesData[riskFreeRatesData['daystomaturity'] == 30].copy()
        rf_monthly = rf_monthly.set_index('date').resample('ME').last().reset_index()
        rf_monthly['date'] = rf_monthly['date'].dt.to_period('M').dt.to_timestamp('M')
        rf_monthly.to_parquet(riskfree_output, index=False)

        # Sort data and label each daily row with its calendar month-end date
        pricesData = pricesData.sort_values(['securityid', 'date']).copy()
        pricesData['month_end'] = pricesData['date'].dt.to_period('M').dt.to_timestamp('M')

        # Compute adjusted close: closeprice * adjustmentfactor2 / last(adjustmentfactor2 per security)
        last_adj2 = pricesData.groupby('securityid')['adjustmentfactor2'].transform('last')
        pricesData['adjustedclose'] = pricesData['closeprice'] * pricesData['adjustmentfactor2'] / last_adj2

        # Compute 20-trading-day lookback return per security
        pricesData['return_month'] = pricesData.groupby('securityid')['adjustedclose'].pct_change(20)
        pricesData['return_month'] = pricesData['return_month'].replace([np.inf, -np.inf], np.nan)

        # Merge monthly risk-free rate onto daily data by calendar month
        pricesData = pricesData.merge(
            rf_monthly[['date', 'yld_pct_monthly']].rename(columns={'date': 'month_end_rf'}),
            left_on='month_end', right_on='month_end_rf', how='left'
        ).drop(columns=['month_end_rf'])

        # Compute excess return = monthly return minus risk-free rate
        pricesData['return_month_excess'] = pricesData['return_month'] - pricesData['yld_pct_monthly']

        # Shift by -20 trading days to get the one-month-ahead excess return (the outcome variable)
        pricesData['return_month_ahead_excess'] = (
            pricesData.groupby('securityid')['return_month_excess'].shift(-20)
        )

        # Compute market cap from closing price times shares outstanding
        pricesData['MCap'] = pricesData['closeprice'] * pricesData['sharesoutstanding']

        # Collapse to one row per security per calendar month (take last trading day values)
        monthly_prices = (
            pricesData
            .groupby(['securityid', 'month_end'])
            .last()
            .reset_index()
            .drop(columns=['date'])
            .rename(columns={'month_end': 'date'})
        )

        # Filter to the relevant date range and save
        monthly_prices = monthly_prices[
            (monthly_prices['date'] >= startDate) & (monthly_prices['date'] <= endDate)
        ]
        pricesData = monthly_prices
        pricesData.to_parquet(prices_output, index=False)

        ########## STUDENT TASK END ##########

        logger.info(f"Processed risk free rates saved to {riskfree_output}")
        logger.info(f"Processed data saved to {prices_output}")

    def mergeData(self):
        # File paths for the data
        prices_filepath = resultsPath + "/SPXConstituentsPricesPrepared.parquet"
        cram_filepath = baseFolder + "/equityZooMeasures.parquet"
        merged_output = resultsPath + "/MergedInputData.parquet"

        # Load the data
        pricesData = pd.read_parquet(prices_filepath)
        cramData = pd.read_parquet(cram_filepath)

        ########## STUDENT TASK START ##########
        # Produce 'mergedData' and save it to merged_output (parquet,
        # index=False). Each row must carry the CRAM measure values for a
        # given (security, date) plus that security's next-month excess
        # return from pricesData.

        # Merge CRAM measures with monthly prices on SecurityID and date
        mergedData = cramData.merge(
            pricesData[['securityid', 'date', 'MCap', 'return_month_ahead_excess']],
            left_on=['SecurityID', 'loctimestamp'],
            right_on=['securityid', 'date'],
            how='inner'
        )

        mergedData = mergedData.drop(columns=['securityid', 'date'])

        ########## STUDENT TASK END ##########

        mergedData.to_parquet(merged_output, index=False)
        logger.info(f"Merged data saved to {merged_output}")

    def getSortingDates(self):

        # File paths
        merged_filepath = resultsPath + "/MergedInputData.parquet"
        filtered_output = resultsPath + "/MergedInputDataFiltered.parquet"
        sorting_dates_output = resultsPath + "/SortingDates.parquet"

        # Load merged data
        mergedData = pd.read_parquet(merged_filepath)

        mergedData['loctimestamp'] = pd.to_datetime(mergedData['loctimestamp'])

        ########## STUDENT TASK START ##########
        # Produce TWO objects:
        #   first_of_month_dates : a pd.Series of the first available trading
        #                          day in each calendar month
        #   filteredData         : mergedData restricted to rows whose
        #                          loctimestamp is one of those days


        all_dates = pd.Series(sorted(mergedData['loctimestamp'].unique()))

        # For each calendar month, take the earliest (first) available date
        first_of_month_dates = (
            all_dates
            .groupby(all_dates.dt.to_period('M'))
            .first()
            .reset_index(drop=True)
        )

        # Keep only rows whose loctimestamp is one of the first-of-month dates
        filteredData = mergedData[mergedData['loctimestamp'].isin(first_of_month_dates.values)].copy()

        ########## STUDENT TASK END ##########

        filteredData = filteredData.reset_index(drop=True)
        first_of_month_dates = first_of_month_dates.reset_index(drop=True)

        # Save the filtered dataframe
        filteredData.to_parquet(filtered_output, index=False)

        # Save the list of first-of-month dates
        first_of_month_dates.to_frame(name="first_of_month").to_parquet(sorting_dates_output, index=False)

        logger.info(f"Filtered dataset saved to {filtered_output}")
        logger.info(f"Sorting dates saved to {sorting_dates_output}")

    def buildSortedPortfolios(self):
        # File paths
        instrument_data_path = resultsPath + "/MergedInputDataFiltered.parquet"
        instrumentData = pd.read_parquet(instrument_data_path)
        sorting_dates_path = resultsPath + "/SortingDates.parquet"
        sortingDates = pd.read_parquet(sorting_dates_path)
        sortingDates = sortingDates["first_of_month"].tolist() # convert to list
        if testmode:
            sortingDates = sortingDates[0:5] # Short list for testing
        logger.info(f'Starting calculation of sorted portfolios: {sortingType}')
        logger.info(f'Sorting variable 1: {sortVariable1}, sorting variable 2: {sortVariable2}, outcome variable: {outcomeVariable}, percentiles variable 1: {sortPercentilesVar1}, percentiles variable 2: {sortPercentilesVar2}')
        logger.info(f'Weighting: {weightType}, filter for portfolios: {filterPortfolios}, maximum possible backtracking: {backtrackingDays} days')
        portfolioWeightsDict = {} # Key: portfolioID, Value: {date: weights}
        outcomeResultsDict = {} # Key: portfolioID, Value: {date: averageOutcome}
        numberOfStocksDict = {} # Number of stocks per portfolio
        breakpointsDict = {} # Breakpoints per date
        missingDataDict = {} # Key: date, Value: {errortype: count}
        missingRebalancingDatesList = [] # List containing missing rebalancing dates
        constituents = instrumentData['instrumentid'].unique().tolist()

        def countNaNs(df, colname): # Function to handle possible duplicate column names when counting errors
            col = df[colname]
            if isinstance(col, pd.DataFrame): # If column name, i.e. variable, comes up more than once
                return col.iloc[:, 0].isna().sum()
            return col.isna().sum()
        
        for date in sortingDates: # Iterate over each date and perform sorting
            workingDate = date
            validDataFound = False
            backtrackAttempts = 0
            missingData = {date: {"instruments_missing": 0, "sortVariable1_nan": 0, "sortVariable2_nan": 0, "marketCap_nan": 0, "outcomeVar_nan": 0, "portfolio_insufficient": 0}}
            if sortingType == "Univariate":
                # This date's slice of (instrument, sort variable, mcap, outcome)
                dateDataframe = instrumentData[
                    (instrumentData['loctimestamp'] == workingDate) &
                    (instrumentData['instrumentid'].isin(constituents))
                ][['instrumentid', sortVariable1, marketCapColumn, outcomeVariable]]
                validDataframe = dateDataframe.dropna(
                    subset=[sortVariable1, marketCapColumn, outcomeVariable]
                ).copy()
                # Also remove rows with infinite values (dropna only catches NaN, not inf)
                validDataframe = validDataframe[
                    np.isfinite(validDataframe[sortVariable1]) &
                    np.isfinite(validDataframe[marketCapColumn]) &
                    np.isfinite(validDataframe[outcomeVariable])
                ]
                if validDataframe.empty:
                    missingRebalancingDatesList.append(date)
                    continue

                # Bookkeeping for missing data (DO NOT MODIFY)
                missingIDs = set(constituents) - set(dateDataframe['instrumentid'])
                missingData[date]["instruments_missing"] += len(missingIDs)
                missingData[date]["sortVariable1_nan"] += int(countNaNs(dateDataframe, sortVariable1))
                missingData[date]["marketCap_nan"]     += int(countNaNs(dateDataframe, marketCapColumn))
                missingData[date]["outcomeVar_nan"]    += int(countNaNs(dateDataframe, outcomeVariable))
                missingDataDict[date] = missingData[date]

                ########## STUDENT TASK START ##########
                # Compute percentile breakpoints for sortVariable1
                sort1Values = validDataframe[sortVariable1].to_numpy()
                breakpoints = [float(np.percentile(sort1Values, p)) for p in sortPercentilesVar1]
                n_portfolios = len(breakpoints) + 1

                # Assign each instrument to a portfolio bucket.
                # pd.cut places a value exactly on a breakpoint into the lower bucket.
                # The boundary dict also adds it to the upper bucket so that instruments
                # on a breakpoint belong to BOTH adjacent buckets, as required.
                bins = [-np.inf] + breakpoints + [np.inf]
                assignments = pd.cut(validDataframe[sortVariable1], bins=bins, labels=range(1, n_portfolios + 1))

                boundary_upper = {k: [] for k in range(1, n_portfolios + 2)}
                for bpi, bpv in enumerate(breakpoints):
                    on_boundary = validDataframe.index[validDataframe[sortVariable1] == bpv].tolist()
                    if on_boundary:
                        boundary_upper[bpi + 2] += on_boundary

                # Determine which portfolios to keep (all by default)
                active_portfolios = list(range(1, n_portfolios + 1))
                if filterPortfolios:
                    active_portfolios = filterPortfolios

                portfolioWeights = {}
                portfolioOutcome = {}

                for k in active_portfolios:
                    extra = boundary_upper.get(k, [])
                    if extra:
                        sub_df = validDataframe[(assignments == k) | validDataframe.index.isin(extra)]
                    else:
                        sub_df = validDataframe[assignments == k]
                    if sub_df.empty:
                        continue
                    instruments = sub_df['instrumentid'].tolist()
                    outcomes = sub_df[outcomeVariable].to_numpy(dtype=float)
                    mcaps = sub_df[marketCapColumn].to_numpy(dtype=float)
                    if weightType == "MarketCapWeighted":
                        total_mcap = mcaps.sum()
                        if total_mcap <= 0:
                            continue
                        weights = {inst: float(mc) / total_mcap for inst, mc in zip(instruments, mcaps)}
                    else:  # EquallyWeighted
                        n_k = len(instruments)
                        weights = {inst: 1.0 / n_k for inst in instruments}
                    key = f"{sortVariable1}_{k}"
                    portfolioWeights[key] = weights
                    portfolioOutcome[key] = float(sum(weights[inst] * out for inst, out in zip(instruments, outcomes)))

                ########## STUDENT TASK END ##########

                # Fold per-date results into the global accumulators (DO NOT MODIFY)
                for portfolioID, weightStructure in portfolioWeights.items():
                    portfolioWeightsDict.setdefault(portfolioID, {})[date] = weightStructure
                for portfolioID, outcome in portfolioOutcome.items():
                    outcomeResultsDict.setdefault(portfolioID, {})[date] = outcome
                breakpointsDict.setdefault(date, {})[sortVariable1] = breakpoints

            elif sortingType in ["BivariateDependent", "BivariateIndependent"]:
                while not validDataFound and backtrackAttempts <= backtrackingDays:
                    # Filter instrumentData to current date and relevant instrument IDs
                    dateDataframe = instrumentData[
                        (instrumentData['instrumentid'].isin(constituents)) &
                        (instrumentData['loctimestamp'] == workingDate)
                    ][['instrumentid', sortVariable1, sortVariable2, marketCapColumn, outcomeVariable]]
                    validDataframe = dateDataframe.dropna(subset=[sortVariable1, sortVariable2, marketCapColumn, outcomeVariable]).copy() # Drop rows with any NaNs in relevant columns
                    # Also remove rows with infinite values (dropna only catches NaN, not inf)
                    validDataframe = validDataframe[
                        np.isfinite(validDataframe[sortVariable1]) &
                        np.isfinite(validDataframe[sortVariable2]) &
                        np.isfinite(validDataframe[marketCapColumn]) &
                        np.isfinite(validDataframe[outcomeVariable])
                    ]
                    # Step 1: Determine values of the sort variables, if available
                    sort1Values = validDataframe[sortVariable1].to_numpy() # Collect values of sortVariable1
                    sort2Values = validDataframe[sortVariable2].to_numpy() # Collect values of sortVariable2
                    if len(sort1Values) > 0 and len(sort2Values) > 0:
                        validDataFound = True
                    else:
                        logger.warning(f"No valid CRAM data found for {workingDate}. Backtracking to the previous date.")
                        backtrackAttempts += 1
                        workingDate = workingDate - pd.Timedelta(days=1)
                if backtrackAttempts > backtrackingDays:
                    logger.error(f"ERROR Failed to find valid CRAM data for {date} after {backtrackingDays} attempts.")
                    missingRebalancingDatesList.append(date)
                    continue # Stop searching and log error for the original date.
                # Count how many expected instruments were not found for the date
                missingIDs = set(constituents) - set(dateDataframe['instrumentid'])
                missingData[date]["instruments_missing"] += len(missingIDs)
                # Count missing values in each column
                missingData[date]["sortVariable1_nan"] += int(countNaNs(dateDataframe, sortVariable1))
                missingData[date]["sortVariable2_nan"] += int(countNaNs(dateDataframe, sortVariable2))
                missingData[date]["marketCap_nan"]     += int(countNaNs(dateDataframe, marketCapColumn))
                missingData[date]["outcomeVar_nan"]    += int(countNaNs(dateDataframe, outcomeVariable))
                missingDataDict[date] = missingData[date]

                if sortingType == "BivariateDependent":
                    ########## STUDENT TASK START ##########
                    n_g1 = len(sortPercentilesVar1) + 1
                    n_g2 = len(sortPercentilesVar2) + 1

                    # Sort by Variable 1 unconditionally. Instruments on a breakpoint
                    # belong to BOTH adjacent buckets (same rule as univariate sort).
                    bp1 = [float(np.percentile(sort1Values, p)) for p in sortPercentilesVar1]
                    bins1 = [-np.inf] + bp1 + [np.inf]
                    vdf = validDataframe.copy()
                    vdf['_group1'] = pd.cut(vdf[sortVariable1], bins=bins1, labels=range(1, n_g1 + 1))
                    breakpointsDict.setdefault(date, {})[sortVariable1] = bp1

                    bnd1 = {k: [] for k in range(1, n_g1 + 2)}
                    for bpi, bpv in enumerate(bp1):
                        on_bnd = vdf.index[vdf[sortVariable1] == bpv].tolist()
                        if on_bnd:
                            bnd1[bpi + 2] += on_bnd

                    portfolioWeights = {}
                    portfolioOutcome = {}
                    skip_date = False


                        # BU KISMA TEKRAR BAK GERCEKTEN ANLAMADIM, SIRALAMA MANTIGI KAFAMI KARISTIRDI
                    for i in range(1, n_g1 + 1):
                        extra_g1 = bnd1.get(i, [])
                        if extra_g1:
                            group1_df = vdf[(vdf['_group1'] == i) | vdf.index.isin(extra_g1)].copy()
                        else:
                            group1_df = vdf[vdf['_group1'] == i].copy()
                        if group1_df.empty:
                            missingRebalancingDatesList.append(date)
                            missingData[date]["portfolio_insufficient"] += 1
                            skip_date = True
                            break
                        # Within each Var1 group, sort by Variable 2 conditionally
                        sort2_in_group = group1_df[sortVariable2].to_numpy(dtype=float)
                        bp2 = [float(np.percentile(sort2_in_group, p)) for p in sortPercentilesVar2]
                        bins2 = [-np.inf] + bp2 + [np.inf]
                        group1_df['_group2'] = pd.cut(group1_df[sortVariable2], bins=bins2, labels=range(1, n_g2 + 1))
                        breakpointsDict.setdefault(date, {})[f"{sortVariable2}_in_{sortVariable1}_{i}"] = bp2

                        bnd2 = {k: [] for k in range(1, n_g2 + 2)}
                        for bpi2, bpv2 in enumerate(bp2):
                            on_bnd2 = group1_df.index[group1_df[sortVariable2] == bpv2].tolist() #WHY???????
                            if on_bnd2:
                                bnd2[bpi2 + 2] += on_bnd2

                        for j in range(1, n_g2 + 1):
                            extra_g2 = bnd2.get(j, [])
                            if extra_g2:
                                sub_df = group1_df[(group1_df['_group2'] == j) | group1_df.index.isin(extra_g2)]
                            else:
                                sub_df = group1_df[group1_df['_group2'] == j]
                            if sub_df.empty:
                                missingRebalancingDatesList.append(date)
                                missingData[date]["portfolio_insufficient"] += 1
                                skip_date = True
                                break
                            key = f"{sortVariable1}_{i}_{sortVariable2}_{j}"
                            instruments = sub_df['instrumentid'].tolist()
                            outcomes = sub_df[outcomeVariable].to_numpy(dtype=float)
                            mcaps = sub_df[marketCapColumn].to_numpy(dtype=float)
                            if weightType == "MarketCapWeighted":
                                total_mcap = mcaps.sum()
                                if total_mcap <= 0:
                                    missingRebalancingDatesList.append(date)
                                    missingData[date]["portfolio_insufficient"] += 1
                                    skip_date = True
                                    break
                                weights = {inst: float(mc) / total_mcap for inst, mc in zip(instruments, mcaps)}
                            else:
                                n_k = len(instruments)
                                weights = {inst: 1.0 / n_k for inst in instruments}
                            portfolioWeights[key] = weights
                            portfolioOutcome[key] = float(sum(weights[inst] * out for inst, out in zip(instruments, outcomes)))
                        if skip_date:
                            break

                    if skip_date:
                        continue

                    ########## STUDENT TASK END ##########

                    # Fold per-date results into the global accumulators (DO NOT MODIFY)
                    missingDataDict[date] = missingData[date]
                    for portfolioID, weightStructure in portfolioWeights.items():
                        portfolioWeightsDict.setdefault(portfolioID, {})[date] = weightStructure
                    for portfolioID, outcome in portfolioOutcome.items():
                        outcomeResultsDict.setdefault(portfolioID, {})[date] = outcome

                elif sortingType == "BivariateIndependent":
                    ########## STUDENT TASK START ##########
                    n_g1 = len(sortPercentilesVar1) + 1
                    n_g2 = len(sortPercentilesVar2) + 1

                    # Sort both variables fully independently using the full cross-section
                    bp1 = [float(np.percentile(sort1Values, p)) for p in sortPercentilesVar1]
                    bp2 = [float(np.percentile(sort2Values, p)) for p in sortPercentilesVar2]
                    bins1 = [-np.inf] + bp1 + [np.inf]
                    bins2 = [-np.inf] + bp2 + [np.inf]

                    vdf = validDataframe.copy()
                    vdf['_group1'] = pd.cut(vdf[sortVariable1], bins=bins1, labels=range(1, n_g1 + 1))
                    vdf['_group2'] = pd.cut(vdf[sortVariable2], bins=bins2, labels=range(1, n_g2 + 1))

                    breakpointsDict.setdefault(date, {})[sortVariable1] = bp1
                    breakpointsDict.setdefault(date, {})[sortVariable2] = bp2

                    bnd1_ind = {k: [] for k in range(1, n_g1 + 2)}
                    for bpi, bpv in enumerate(bp1):
                        on_bnd = vdf.index[vdf[sortVariable1] == bpv].tolist()
                        if on_bnd:
                            bnd1_ind[bpi + 2] += on_bnd

                    bnd2_ind = {k: [] for k in range(1, n_g2 + 2)}
                    for bpi, bpv in enumerate(bp2):
                        on_bnd = vdf.index[vdf[sortVariable2] == bpv].tolist()
                        if on_bnd:
                            bnd2_ind[bpi + 2] += on_bnd

                    portfolioWeights = {}
                    portfolioOutcome = {}
                    skip_date = False

                    for i in range(1, n_g1 + 1):
                        for j in range(1, n_g2 + 1):
                            mi = (vdf['_group1'] == i) | vdf.index.isin(bnd1_ind.get(i, []))
                            mj = (vdf['_group2'] == j) | vdf.index.isin(bnd2_ind.get(j, []))
                            sub_df = vdf[mi & mj]
                            if sub_df.empty:
                                missingRebalancingDatesList.append(date)
                                missingData[date]["portfolio_insufficient"] += 1
                                skip_date = True
                                break
                            key = f"{sortVariable1}_{i}_{sortVariable2}_{j}"
                            instruments = sub_df['instrumentid'].tolist()
                            outcomes = sub_df[outcomeVariable].to_numpy(dtype=float)
                            mcaps = sub_df[marketCapColumn].to_numpy(dtype=float)
                            if weightType == "MarketCapWeighted":
                                total_mcap = mcaps.sum()
                                if total_mcap <= 0:
                                    missingRebalancingDatesList.append(date)
                                    missingData[date]["portfolio_insufficient"] += 1
                                    skip_date = True
                                    break
                                weights = {inst: float(mc) / total_mcap for inst, mc in zip(instruments, mcaps)}
                            else:
                                n_k = len(instruments)
                                weights = {inst: 1.0 / n_k for inst in instruments}
                            portfolioWeights[key] = weights
                            portfolioOutcome[key] = float(sum(weights[inst] * out for inst, out in zip(instruments, outcomes)))
                        if skip_date:
                            break

                    if skip_date:
                        continue

                    ########## STUDENT TASK END ##########

                    # Fold per-date results into the global accumulators (DO NOT MODIFY)
                    missingDataDict[date] = missingData[date]
                    for portfolioID, weightStructure in portfolioWeights.items():
                        portfolioWeightsDict.setdefault(portfolioID, {})[date] = weightStructure
                    for portfolioID, outcome in portfolioOutcome.items():
                        outcomeResultsDict.setdefault(portfolioID, {})[date] = outcome

        # Sort portfolio results by date ascending for each portfolio
        sortedPortfolioWeightsDict = {
            portfolioID: dict(sorted(weightsByDate.items()))
            for portfolioID, weightsByDate in portfolioWeightsDict.items()
            }
        # Sort outcome results by date ascending for each portfolio
        sortedOutcomeResultsDict = {
            portfolioID: dict(sorted(outcomeByDate.items()))
            for portfolioID, outcomeByDate in outcomeResultsDict.items()
            }
        # Sort missingData by date ascending
        missingDataDict = dict(sorted(missingDataDict.items(), key=lambda item: item[0]))
        # Sort breakpoints by date ascending
        breakpointsDict = dict(sorted(breakpointsDict.items(), key=lambda item: item[0]))
        # Count number of stocks
        for portfolioID, portfolioWeights in sortedPortfolioWeightsDict.items():
            if portfolioID not in numberOfStocksDict:
                numberOfStocksDict[portfolioID] = {}
            for date, weights in portfolioWeights.items():
                numberOfStocksDict[portfolioID][date] = len(weights)
        
        # PRINT RESULTS TO LOG
        if log_results:
            # logger.info("------Portfolios:-------")
            # for portfolio, weightStructure in sortedPortfolioWeightsDict.items():
            #     logger.info(f'Portfolio {portfolio}:')
            #     for date, weights in weightStructure.items():
            #         logger.info(f'Date {date}: Something')
            logger.info("------Outcome:-------")
            for portfolio, outcomeStruct in sortedOutcomeResultsDict.items():
                logger.info(f'Portfolio {portfolio}:')
                for date, outcome in outcomeStruct.items():
                    logger.info(f'Date {date}: {outcome}')
            logger.info("------Missing Data:------")
            for date, dataInfo in missingDataDict.items():
                logger.info(date)
                logger.info(dataInfo)
            logger.info("------Missing rebalancing dates:------")
            logger.info(missingRebalancingDatesList)
            logger.info("------Number of stocks:------")
            for portfolio, nstocksStruct in numberOfStocksDict.items():
                logger.info(f'Portfolio {portfolio}:')
                for date, nstocks in nstocksStruct.items():
                    logger.info(f'Date {date}: {nstocks}')
            logger.info("------Breakpoints:------")
            for date, struct in breakpointsDict.items():
                logger.info(f'date: {date}')
                for var, bps in struct.items():
                    logger.info(f'variable: {var}')
                    logger.info(f'breakpoints: {bps}')
        # Dictionary of variables to save
        variables_to_save = {
            "sortedPortfolioResults": sortedPortfolioWeightsDict,
            "sortedOutcomeResults": sortedOutcomeResultsDict,
            "numberOfStocks": numberOfStocksDict,
            "breakpointsDict": breakpointsDict,
            "missingDataDict": missingDataDict,
            "missingRebalancingDatesList": missingRebalancingDatesList
        }
        # Loop through dictionary and save each variable as a .pkl file
        for var_name, var_value in variables_to_save.items():
            file_path = os.path.join(resultsPath, f"{var_name}.pkl")
            with open(file_path, "wb") as f:
                pickle.dump(var_value, f)
            logger.info(f"Saved {var_name} to {file_path}")
        logger.info('Completed calculation of sorted portfolios.')

    def convertOutcome(self):
        # Path to the input pickle file
        filtered_output = resultsPath + "/sortedOutcomeResults.pkl"
        
        # Load the sortedOutcomeResults from pickle
        with open(filtered_output, 'rb') as f:
            sortedOutcomeResults = pickle.load(f)
        
        # Create a list to store all the dates (we assume the dates are the same for all portfolios)
        dates = list(next(iter(sortedOutcomeResults.values())).keys())

        # Create a dictionary where keys are portfolio names and values are the corresponding outcome values for each date
        portfolio_data = {}
        
        for portfolio, outcomeStruct in sortedOutcomeResults.items():
            # Create a list of outcomes for the given portfolio, aligned with the dates
            portfolio_data[portfolio] = [outcomeStruct.get(date, None) for date in dates]

        # Convert the dictionary into a DataFrame
        outcome_df = pd.DataFrame(portfolio_data, index=dates)
        
        # Reset index so 'date' becomes a column in the DataFrame
        outcome_df.reset_index(inplace=True)
        outcome_df.rename(columns={'index': 'date'}, inplace=True)

        # Save the DataFrame to a Parquet file
        outcome_df.to_parquet(resultsPath + "/PortfoliosOutcome.parquet", index=False)
        logger.info("PortfoliosOutcome.parquet has been saved.")

    def calcDiffPortfolios(self):
        # Load the PortfoliosOutcome.parquet file
        portfolios_outcome_filepath = resultsPath + "/PortfoliosOutcome.parquet"
        outcome_df = pd.read_parquet(portfolios_outcome_filepath)

        if sortingType == "Univariate":
            # Determine the portfolio names for the highest and lowest portfolios based on sortVariable1
            high_portfolio = f"{sortVariable1}_{len(sortPercentilesVar1) + 1}"  # Highest portfolio
            low_portfolio = f"{sortVariable1}_1"  # Lowest portfolio
            
            # Check if the high and low portfolios exist in the columns
            if high_portfolio not in outcome_df.columns or low_portfolio not in outcome_df.columns:
                logger.error(f"Portfolio columns {high_portfolio} or {low_portfolio} not found in the DataFrame.")
                return
            
            # Add the new column: sortVariable1_Diff (difference between high and low portfolio)
            outcome_df[f"{sortVariable1}_Diff"] = outcome_df[high_portfolio] - outcome_df[low_portfolio]

        elif sortingType in ("BivariateDependent", "BivariateIndependent"):
            logger.info(f"Calculating generalized Diff and Average logic for bivariate sorting: {sortingType}")
            
            # 1. Determine group lengths dynamically from notebook parameters
            n_g1 = len(sortPercentilesVar1) + 1 # Number of portfolios along Sort Variable 1 (rows)
            n_g2 = len(sortPercentilesVar2) + 1 # Number of portfolios along Sort Variable 2 (columns)
            
            # Helper to generate the exact base portfolio string keys
            def get_col(i, j):
                return f"{sortVariable1}_{i}_{sortVariable2}_{j}"

            # --- A. ROW-WISE MARGINAL CALCULATIONS (Across Sort Variable 2 columns) ---
            # For each category of Var 1, compute the high-low difference and row average
            for i in range(1, n_g1 + 1):
                row_cols = [get_col(i, j) for j in range(1, n_g2 + 1)]
                
                # Diff across Var 2 = High - Low (e.g., group n_g2 - group 1)
                outcome_df[f"{sortVariable1}_{i}_{sortVariable2}_Diff"] = outcome_df[get_col(i, n_g2)] - outcome_df[get_col(i, 1)]
                # Avg across Var 2 = Row Mean
                outcome_df[f"{sortVariable1}_{i}_{sortVariable2}_Avg"] = outcome_df[row_cols].mean(axis=1)

            # --- B. COLUMN-WISE MARGINAL CALCULATIONS (Down Sort Variable 1 rows) ---
            # For each category of Var 2, compute the high-low difference and column average
            for j in range(1, n_g2 + 1):
                col_cols = [get_col(i, j) for i in range(1, n_g1 + 1)]
                
                # Diff down Var 1 = High - Low (e.g., group n_g1 - group 1)
                outcome_df[f"{sortVariable1}_Diff_{sortVariable2}_{j}"] = outcome_df[get_col(n_g1, j)] - outcome_df[get_col(1, j)]
                # Avg down Var 1 = Column Mean
                outcome_df[f"{sortVariable1}_Avg_{sortVariable2}_{j}"] = outcome_df[col_cols].mean(axis=1)

            # --- C. CROSS-OVER INTERSECTIONS (The Matrix Outer Corners) ---
            # 1. Sort Variable 1 Diff pooled across Sort Variable 2 Average
            var1_diff_cols = [f"{sortVariable1}_Diff_{sortVariable2}_{j}" for j in range(1, n_g2 + 1)]
            outcome_df[f"{sortVariable1}_Diff_{sortVariable2}_Avg"] = outcome_df[var1_diff_cols].mean(axis=1)
            
            # 2. Sort Variable 2 Diff pooled across Sort Variable 1 Average
            var2_diff_cols = [f"{sortVariable1}_{i}_{sortVariable2}_Diff" for i in range(1, n_g1 + 1)]
            outcome_df[f"{sortVariable1}_Avg_{sortVariable2}_Diff"] = outcome_df[var2_diff_cols].mean(axis=1)

            # 3. Grand Total Sample Mean Portfolio (Average of Average)
            all_base_cols = [get_col(i, j) for i in range(1, n_g1 + 1) for j in range(1, n_g2 + 1)]
            outcome_df[f"{sortVariable1}_Avg_{sortVariable2}_Avg"] = outcome_df[all_base_cols].mean(axis=1)

            # 4. Difference-in-Difference Portfolio (Double Diff)
            # Implements the canonical (D - C) - (B - A) algebraic structure from Table 5.12
            A = outcome_df[get_col(1, 1)]         # Low Var1, Low Var2
            B = outcome_df[get_col(1, n_g2)]      # Low Var1, High Var2
            C = outcome_df[get_col(n_g1, 1)]      # High Var1, Low Var2
            D = outcome_df[get_col(n_g1, n_g2)]   # High Var1, High Var2
            
            outcome_df[f"{sortVariable1}_Diff_{sortVariable2}_Diff"] = (D - C) - (B - A)

        # Save the resulting DataFrame to a new Parquet file
        output_filepath = resultsPath + "/PortfoliosOutcomePlusDiff.parquet"
        outcome_df.to_parquet(output_filepath, index=False)
        
        logger.info(f"Generalized resulting DataFrame with structural columns saved to {output_filepath}")

    def performStatTest(self):
        # Load the PortfoliosOutcomePlusDiff.parquet file
        portfolios_outcome_filepath = resultsPath + "/PortfoliosOutcomePlusDiff.parquet"
        outcome_df = pd.read_parquet(portfolios_outcome_filepath)

        # Exclude the 'date' column and work with portfolio columns
        portfolio_columns = [col for col in outcome_df.columns if col != 'date']

        # Initialize a list to hold the statistics (Avg, StdErr, t-stat, p-value)
        stats_dict = {
            "Average": [],
            "StdErr": [],
            "t-stat": [],
            "p-value": []
        }

        for portfolio in portfolio_columns:
            timeseries = outcome_df[portfolio]

            ########## STUDENT TASK START ##########
            # For this portfolio's time series of returns, compute:
            #   avg     : the unconditional mean
            #   std_err : the standard error of the mean
            #   t_stat  : the Newey-West (HAC) t-statistic for mean = 0,
            #             using `lags` lags (a module-level parameter)
            #   p_value : the two-sided p-value associated with t_stat

            y = timeseries.dropna().to_numpy(dtype=float)
            if len(y) == 0:
                avg = np.nan
                std_err = np.nan
                t_stat = np.nan
                p_value = np.nan
            else:
                avg = float(np.mean(y))
                # Conventional standard error: std / sqrt(n)
                std_err = float(np.std(y, ddof=1) / np.sqrt(len(y)))
                # Newey-West (HAC) t-stat and p-value with lags=6
                X = np.ones(len(y))
                result = OLS(y, X).fit(cov_type='HAC', cov_kwds={'maxlags': lags})
                t_stat = float(result.tvalues[0])
                p_value = float(result.pvalues[0])

            ########## STUDENT TASK END ##########

            stats_dict["Average"].append(avg)
            stats_dict["StdErr"].append(std_err)
            stats_dict["t-stat"].append(t_stat)
            stats_dict["p-value"].append(p_value)

        # Create a DataFrame from the statistics dictionary
        stats_df = pd.DataFrame(stats_dict, index=portfolio_columns)

        # Save the resulting DataFrame to a new Parquet file
        output_filepath = resultsPath + "/PortfoliosOutcomePlusDiffTest.parquet"
        stats_df.to_parquet(output_filepath, index=True)

        logger.info(f"Statistical test results saved to {output_filepath}")

#--------------------
#-----Execution------
#--------------------

# Create object and run all steps one after the other
ps = PortfolioSortsPipeline()
ps.start()
ps.clearSolutionsFolder()
ps.calcMonthlyExcessReturns()
ps.mergeData()
ps.getSortingDates()
ps.buildSortedPortfolios()
ps.convertOutcome()
ps.calcDiffPortfolios()
ps.performStatTest()
ps.end()





2026-06-28 18:28:18.189 | INFO     | __main__:start:50 - Start Time: 2026-06-28 18:28:18
2026-06-28 18:28:18.190 | INFO     | __main__:start:51 - Start portfolio sorts pipeline...
2026-06-28 18:28:18.190 | INFO     | __main__:clearSolutionsFolder:54 - Clearing solution folder at /Users/denizilicali/Downloads/PS4-2/Data/solution...
2026-06-28 18:28:19.057 | INFO     | __main__:calcMonthlyExcessReturns:151 - Processed risk free rates saved to /Users/denizilicali/Downloads/PS4-2/Data/solution/RiskFreeRatesFilteredMonthly.parquet
2026-06-28 18:28:19.057 | INFO     | __main__:calcMonthlyExcessReturns:152 - Processed data saved to /Users/denizilicali/Downloads/PS4-2/Data/solution/SPXConstituentsPricesPrepared.parquet
2026-06-28 18:28:19.650 | INFO     | __main__:mergeData:183 - Merged data saved to /Users/denizilicali/Downloads/PS4-2/Data/solution/MergedInputData.parquet
2026-06-28 18:28:20.383 | INFO     | __main__:getSortingDates:229 - Filtered dataset saved to /Users/denizilicali/Download